# **BERT Fine-Tuning for Sentiment Analysis**

In [ ]:
!pip install -U transformers

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

# **Core Workflow**
Dataset
   ↓
Tokenizer
   ↓
Tokenized Dataset
   ↓
Model
   ↓
Trainer
   ↓
Fine-Tuned Model
   ↓
Inference

# **Sentiment Analysis with BERT**
We will:

-Load dataset
-Tokenize
-Fine-tune BERT
-Evaluate
-Save model
-Load model
-Predict new text

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

In [ ]:
print(dataset["train"][0])

# Step 2 — Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Tokenization Example
text = "I love Hugging Face"

tokens = tokenizer(text)

print(tokens)

Step 3 — Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
# Apply tokenization:

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Why truncation=True?

Transformer models have:

fixed context windows

BERT limit:

512 tokens

Long text must be truncated.

# **Step 4 — Load Model**

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Step 5 — TrainingArguments

This controls training.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    # evaluation
    #evaluation_strategy="epoch",

    # saving
    save_strategy="epoch",

    # hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,

    # optimization
    weight_decay=0.01,

    # mixed precision
    fp16=True,

    # logging
    logging_steps=50,

    # checkpointing
    save_total_limit=2
)

# Step 6 — Evaluation Metrics

In [ ]:
!pip install evaluate

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

# Step 7 — Trainer API

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./sentiment_model")

tokenizer.save_pretrained("./sentiment_model")

# Load the model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./sentiment_model")
tokenizer = AutoTokenizer.from_pretrained("./sentiment_model")

# Prediction

In [ ]:
text = "This movie was amazing!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=-1)

print(prediction)